# ReAct Agent Intro

Build a ReAct agent with web, Wikipedia, and custom tools using LangGraph.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import HumanMessage
from langchain_openrouter import ChatOpenRouter
from langchain.agents import create_agent

env_path = next(
    (path / ".env" for path in (Path.cwd(), *Path.cwd().parents) if (path / ".env").exists()),
    Path.cwd() / ".env",
)
load_dotenv(env_path, override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY not found")

part3_dir = Path.cwd()
if not (part3_dir / "init_db.py").exists() and (part3_dir / "Part_3" / "init_db.py").exists():
    part3_dir = part3_dir / "Part_3"
if str(part3_dir) not in sys.path:
    sys.path.insert(0, str(part3_dir))

llm = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0, max_tokens=500)
print("OpenRouter model ready:", llm.model_name)

In [ ]:
search_tool = DuckDuckGoSearchRun(
    description="Search the web for current information."
)

wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(),
    description="Search Wikipedia for encyclopedic information.",
)


@tool
def enterprise_tool(query: str) -> str:
    """Send a message to the internal enterprise system."""
    return f"Enterprise system received: {query}"


tools = [search_tool, wikipedia_tool, enterprise_tool]
print("Tools:", [tool.name for tool in tools])

In [ ]:
react_agent = create_agent(llm, tools=tools)

response = react_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Use enterprise_tool to send: Team lunch is at 1 PM in the cafeteria."
            )
        ]
    }
)

response["messages"][-1].content